# Non-Speech Audio Recommendation
This notebook selects 3 random audio clips tagged as non-speech and finds their top 10 most similar non-speech audio clips using the CLAP FAISS index.

**Note:** Rendering too many audio files can freeze the notebook. Audio playback is limited to the first 3 queries.

In [ ]:
import os
import json
import faiss
import glob
import random
import numpy as np
import pandas as pd
from IPython.display import display, Audio


In [ ]:
# --- Configuration ---
PROJECT_ROOT = os.path.abspath('..')
INDEX_DIR = os.path.join(PROJECT_ROOT, "data_index")
INDEX_FILENAME = "clap_embeddings.faiss"
METADATA_FILENAME = "clap_metadata.json"

In [ ]:
# --- Find Audio Files ---
AUDIO_DIR = os.path.join(PROJECT_ROOT, "data", "audio")
audio_files = glob.glob(os.path.join(AUDIO_DIR, "**", "*.*"), recursive=True)
audio_path_map = {os.path.basename(f): f for f in audio_files}
print(f"Found {len(audio_path_map)} audio files in {AUDIO_DIR}")


In [ ]:
# --- Load Index and Metadata ---
print("Loading metadata...")
with open(os.path.join(INDEX_DIR, METADATA_FILENAME), 'r') as f:
    metadata = json.load(f)

print("Loading FAISS index...")
index_path = os.path.join(INDEX_DIR, INDEX_FILENAME)
index = faiss.read_index(index_path)

print(f"Loaded {len(metadata)} metadata records and {index.ntotal} index vectors.")

In [ ]:
# --- Filter Metadata ---
# Create a dictionary for fast lookup
metadata_dict = {item['id']: item for item in metadata}

# Get all non-speech IDs
non_speech_items = [item for item in metadata if not item['is_speech']]
print(f"Found {len(non_speech_items)} non-speech audio clips.")

In [ ]:
# --- Select 3 Random Non-Speech Clips ---
random.seed(42)
query_items = random.sample(non_speech_items, 3)
query_ids = [item['id'] for item in query_items]

print("Selected 3 random non-speech clips for querying.")

In [ ]:
# --- Retrieve Embeddings ---
# We reconstruct the embeddings for the 20 queries from the FAISS index.
try:
    query_embeddings = np.vstack([index.reconstruct(int(id)) for id in query_ids])
except Exception as e:
    print(f"Failed to reconstruct embeddings: {e}")

In [ ]:
# --- Perform Search and Filter ---
TOP_K_SIMILAR = 10
SEARCH_K = 100 # Fetch more results to allow for filtering non-speech items later

distances, indices = index.search(query_embeddings, SEARCH_K)

results = []
for i, q_item in enumerate(query_items):
    q_id = q_item['id']
    q_filename = q_item['filename']
    
    valid_matches = []
    for rank, (match_id, dist) in enumerate(zip(indices[i], distances[i])):
        # Skip the query itself
        if match_id == q_id:
            continue
            
        match_meta = metadata_dict.get(match_id)
        # Check if the matched item is non-speech
        if match_meta and not match_meta['is_speech']:
            valid_matches.append({
                'filename': match_meta['filename'],
                'similarity_score': float(dist)
            })
            
        # Stop once we have 10 valid recommendations
        if len(valid_matches) == TOP_K_SIMILAR:
            break
            
    results.append({
        'query_filename': q_filename,
        'recommendations': valid_matches
    })


In [ ]:
# --- Display Results ---
# LIMIT audio displays to first 3 queries to prevent notebook hangs
DISPLAY_LIMIT = 3

for i, res in enumerate(results):
    q_filename = res["query_filename"]
    print(f"\n{'='*50}\nQuery: {q_filename}\n{'='*50}")
    
    show_audio = i < DISPLAY_LIMIT
    
    # Play query audio
    q_path = audio_path_map.get(q_filename)
    if q_path:
        if show_audio:
            display(Audio(q_path))
        else:
            print(f"[Audio playback skipped for {q_filename}]")
    else:
        print("Audio file not found for query.")
        
    print("\nTop 10 Non-Speech Recommendations:")
    
    for j, rec in enumerate(res["recommendations"], 1):
        rec_filename = rec["filename"]
        score = rec["similarity_score"]
        print(f"\n{j}. Match: {rec_filename} (Score: {score:.4f})")
        
        rec_path = audio_path_map.get(rec_filename)
        if rec_path:
            if show_audio:
                display(Audio(rec_path))
        else:
            print("Audio file not found.")
            
    if i == DISPLAY_LIMIT - 1:
        print(f"\n[INFO] Stopped rendering Audio widgets after {DISPLAY_LIMIT} queries to prevent notebook instability.")
